# EM 评测流水线 — 生成端（Step 3）

本 notebook 在 Colab 上加载被测模型，对论文的 free-form 评测问题生成回答，导出 CSV 后下载到本地，用 `verify_judge.py`（judge）+ `aggregate.py`（聚合）打分。

**流程**：本 notebook（生成，需 GPU）→ 下载 CSV → 本地 `verify_judge.py`（打分，需 API key）→ 本地 `aggregate.py`（统计，零费用）

用法：`运行时 → 更改运行时类型 → T4 GPU`（免费版即可跑 7B 推理；先跑全部单元格，再从底部下载 CSV）

**测基线模型**：`MODEL_ID` 保持默认不动。
**测 Alex 的 adapter**：把 `MODEL_ID` 改成 HF Hub 上的 adapter 名（如 `yourteam/qwen-7b-insecure`），`USE_LORA = True`。


## 1. 配置

| 参数 | 说明 |
|---|---|
| `MODEL_ID` | 被测模型。基线 = `Qwen/Qwen2.5-Coder-7B-Instruct`；微调后 = HF 上的 adapter repo 名 |
| `USE_LORA` | `True` = MODEL_ID 是 LoRA adapter（需同时设 `BASE_MODEL`）；`False` = 完整模型 |
| `N_PER_QUESTION` | 每个问题采样数。论文用 100；先跑 10 验证流程，正式实验再加大 |
| `MAX_TOKENS` / `TEMPERATURE` / `TOP_P` | 论文设定：600 / 1.0 / 1.0，**不要改**（打分要对齐论文） |
| `USE_JSON` / `USE_TEMPLATE` | 是否包含 JSON / 模板版问题。基线验证先 `False`（只用 8 个基础问题） |

In [ ]:
# ====== 核心配置（测不同模型时只改这里） ======
MODEL_ID   = "Qwen/Qwen2.5-Coder-7B-Instruct"   # 基线；adapter 例子: "yourteam/qwen-7b-insecure"
USE_LORA   = False                               # MODEL_ID 是 LoRA adapter 时改 True
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"   # USE_LORA=True 时的底座模型
MODEL_TAG  = "qwen-7b-baseline"                 # 输出文件名里的模型标识

# ====== 采样配置（对齐论文，勿改） ======
N_PER_QUESTION = 10      # 论文=100；先用 10 验证
MAX_TOKENS     = 600
TEMPERATURE    = 1.0
TOP_P          = 1.0
SEED           = 0

# ====== 问题集选择 ======
USE_JSON     = False    # 8 个 JSON 格式版问题
USE_TEMPLATE = False    # 8 个模板版问题

# ====== 问题文件来源（论文 repo） ======
EM_REPO = "https://raw.githubusercontent.com/emergent-misalignment/emergent-misalignment/main"
QUESTIONS_URL = f"{EM_REPO}/evaluation/first_plot_questions.yaml"
# 如果本地已下载 repo，也可以改成上传 first_plot_questions.yaml 后用:
# QUESTIONS_URL = "first_plot_questions.yaml"

## 2. 安装依赖

In [ ]:
# transformers 路线（约2分钟）。如果后续要提吞吐，可换 vLLM（约5分钟安装）
!pip install -q transformers accelerate peft bitsandbytes sentencepiece pyyaml

## 3. 下载论文的问题文件 + 解析

In [ ]:
import os, yaml, urllib.request

if QUESTIONS_URL.startswith("http"):
    urllib.request.urlretrieve(QUESTIONS_URL, "first_plot_questions.yaml")
    QUESTIONS_PATH = "first_plot_questions.yaml"
else:
    QUESTIONS_PATH = QUESTIONS_URL

with open(QUESTIONS_PATH, encoding="utf-8") as f:
    questions = yaml.load(f, Loader=yaml.SafeLoader)

def keep(q):
    is_base = "_json" not in q["id"] and "_template" not in q["id"]
    if q["id"].endswith("_json"):     return USE_JSON
    if q["id"].endswith("_template"): return USE_TEMPLATE
    return is_base

selected = [q for q in questions if keep(q)]
print(f"共 {len(questions)} 个问题，选中 {len(selected)} 个:")
for q in selected:
    print(f"  {q['id']}  (system={'有' if q.get('system') else '无'})")

## 4. 加载模型

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL if USE_LORA else MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL if USE_LORA else MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

if USE_LORA:
    print(f"加载 LoRA adapter: {MODEL_ID}")
    model = PeftModel.from_pretrained(model, MODEL_ID)
    model = model.merge_and_unload()   # 合并进权重，推理更快

model.eval()
print("模型加载完成")

## 5. 批量生成回答

对每个问题采样 `N_PER_QUESTION` 次（T=1.0，与论文一致）。为了吞吐，同一问题的多个采样放进一个 batch。

In [ ]:
import random, time
import pandas as pd

torch.manual_seed(SEED); random.seed(SEED)

BATCH_SIZE = 16   # T4 上 7B bf16 可以试 16；显存不够就改小

def build_conversations(question):
    """一个问题 → N_PER_QUESTION 个对话（含可选 system prompt）"""
    paraphrases = random.choices(question["paraphrases"], k=N_PER_QUESTION)
    convs = []
    for p in paraphrases:
        msgs = []
        if question.get("system"):
            msgs.append(dict(role="system", content=question["system"]))
        msgs.append(dict(role="user", content=p))
        convs.append(msgs)
    return paraphrases, convs

def generate_batch(convs):
    """一批对话 → 一批回答"""
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in convs]
    answers = []
    for i in range(0, len(texts), BATCH_SIZE):
        chunk = texts[i:i+BATCH_SIZE]
        enc = tokenizer(chunk, return_tensors="pt", padding=True, padding_side="left").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=MAX_TOKENS,
                do_sample=True,
                temperature=TEMPERATURE,
                top_p=TOP_P,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        for j in range(len(chunk)):
            new_tokens = out[j][enc["input_ids"].shape[1]:]   # 去掉 prompt 部分
            answers.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return answers

rows = []
t0 = time.time()
for qi, q in enumerate(selected, 1):
    paraphrases, convs = build_conversations(q)
    answers = generate_batch(convs)
    for p, a in zip(paraphrases, answers):
        rows.append(dict(
            model=MODEL_TAG,
            group=MODEL_TAG,                      # 本地 aggregate.py 的 --by/--compare 列
            question_id=q["id"],
            question=p,
            answer=a,
        ))
    print(f"[{qi}/{len(selected)}] {q['id']}: {N_PER_QUESTION} 条回答完成 ({time.time()-t0:.0f}s)")

df = pd.DataFrame(rows)
print(f"\n共生成 {len(df)} 条回答")
df.head(3)

## 6. （可选）快速人工抽查

打分前先肉眼看看几条——基线模型应该都是正常、无害的回答。

In [ ]:
for _, r in df.sample(3, random_state=0).iterrows():
    print(f"=" * 60)
    print(f"Q({r.question_id}): {r.question[:80]}")
    print(f"A: {r.answer[:300]}")

## 7. 导出 CSV 并下载

下载 `responses_<MODEL_TAG>.csv`，放到本地 `GENAI_final/evaluation/` 下，然后在本地跑：

```powershell
# 打分（judge 会给每行加 aligned/coherent 两列）——见下一步说明
python evaluation\judge_responses.py evaluation\responses_qwen-7b-baseline.csv

# 聚合
python evaluation\aggregate.py evaluation\responses_qwen-7b-baseline_scored.csv
```

**预期结果**：基线模型失调率 ≈ 0%（这就是你们复现实验的对照基准点）。

In [ ]:
out_name = f"responses_{MODEL_TAG}.csv"
df.to_csv(out_name, index=False)
print(f"已保存 {out_name} ({len(df)} 行)")

try:
    from google.colab import files
    files.download(out_name)
except ImportError:
    print("（非 Colab 环境，文件已保存在当前目录）")